In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/features.csv", parse_dates=["GAME_DATE"])

print(df.shape)
print(df.columns.tolist())
df.head(3)

(11896, 18)
['GAME_ID', 'GAME_DATE', 'SEASON', 'HOME_TEAM_ID', 'HOME_TEAM', 'HOME_PTS', 'AWAY_TEAM_ID', 'AWAY_TEAM', 'AWAY_PTS', 'HOME_WIN', 'HOME_PTS_ROLL10', 'HOME_PTS_ALLOWED_ROLL10', 'HOME_WIN_ROLL10', 'HOME_REST_DAYS', 'AWAY_PTS_ROLL10', 'AWAY_PTS_ALLOWED_ROLL10', 'AWAY_WIN_ROLL10', 'AWAY_REST_DAYS']


,GAME_ID,GAME_DATE,SEASON,HOME_TEAM_ID,HOME_TEAM,HOME_PTS,AWAY_TEAM_ID,AWAY_TEAM,AWAY_PTS,HOME_WIN,HOME_PTS_ROLL10,HOME_PTS_ALLOWED_ROLL10,HOME_WIN_ROLL10,HOME_REST_DAYS,AWAY_PTS_ROLL10,AWAY_PTS_ALLOWED_ROLL10,AWAY_WIN_ROLL10,AWAY_REST_DAYS
0,21500070,2015-11-05,2015-16,1610612741,CHI,104,1610612760,OKC,98,1,100.6,102.0,0.6,2.0,114.2,109.6,0.6,1.0
1,21500074,2015-11-05,2015-16,1610612757,POR,115,1610612763,MEM,96,1,101.6,99.6,0.6,1.0,92.2,101.6,0.6,2.0
2,21500084,2015-11-06,2015-16,1610612758,SAC,110,1610612745,HOU,116,0,106.2,112.0,0.2,2.0,99.0,109.0,0.4,2.0


In [3]:
cutoff = pd.Timestamp("2022-08-01")

train = df[df["GAME_DATE"] < cutoff]
test = df[df["GAME_DATE"] >= cutoff]

print("Train games:", len(train), "| Test games:", len(test))
print("Train dates:", train["GAME_DATE"].min().date(), "to", train["GAME_DATE"].max().date())
print("Test dates: ", test["GAME_DATE"].min().date(), "to", test["GAME_DATE"].max().date())

Train games: 8211 | Test games: 3685
Train dates: 2015-11-05 to 2022-04-10
Test dates:  2022-10-18 to 2025-04-13


In [4]:
feature_cols = [
    "HOME_PTS_ROLL10", "HOME_PTS_ALLOWED_ROLL10", "HOME_WIN_ROLL10", "HOME_REST_DAYS",
    "AWAY_PTS_ROLL10", "AWAY_PTS_ALLOWED_ROLL10", "AWAY_WIN_ROLL10", "AWAY_REST_DAYS",
]
target_col = "HOME_WIN"

assert "HOME_PTS" not in feature_cols and "AWAY_PTS" not in feature_cols
assert target_col not in feature_cols

train = train.dropna(subset=feature_cols)
test = test.dropna(subset=feature_cols)

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

In [5]:
from sklearn.metrics import accuracy_score, log_loss
import numpy as np

base_rate = y_train.mean()

base_acc = (y_test == 1).mean() if base_rate >= 0.5 else (y_test == 0).mean()
base_probs = np.full(len(y_test), base_rate)
base_ll = log_loss(y_test, base_probs)

print(f"Train home win rate: {base_rate:.4f}")
print(f"Baseline accuracy (always pick home): {base_acc:.4f}")
print(f"Baseline log loss: {base_ll:.4f}")

Train home win rate: 0.5709
Baseline accuracy (always pick home): 0.5560
Baseline log loss: 0.6873


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

acc = accuracy_score(y_test, preds)
ll = log_loss(y_test, probs)
print(f"Logistic accuracy: {acc:.4f} (baseline {base_acc:.4f})")
print(f"Logistic log loss: {ll:.4f} (baseline {base_ll:.4f})")

Logistic accuracy: 0.6353 (baseline 0.5560)
Logistic log loss: 0.6389 (baseline 0.6873)


In [7]:
coefs = pd.Series(model[-1].coef_[0], index=feature_cols).sort_values()
print(coefs)

HOME_PTS_ALLOWED_ROLL10   -0.404474
AWAY_PTS_ROLL10           -0.386502
AWAY_REST_DAYS            -0.093170
AWAY_WIN_ROLL10           -0.076829
HOME_REST_DAYS             0.088842
HOME_WIN_ROLL10            0.112725
AWAY_PTS_ALLOWED_ROLL10    0.301897
HOME_PTS_ROLL10            0.403715
dtype: float64
